# ♻️ Entrenamiento YOLO11 — Detección de Residuos

Notebook para **Google Colab** (GPU T4 gratuita). Entrena un detector de **6 clases** de residuos:
`Trash`, `cardboard`, `glass`, `metal`, `paper`, `plastic`.

**Proyecto:** `deteccion-residuos-yolo`

---
### ⚙️ ANTES DE EMPEZAR (importante)
1. Menú **Entorno de ejecución → Cambiar tipo de entorno → Acelerador por hardware: GPU (T4)**.
2. Sube tu archivo `yolo-v8-trash-detection-EE4016.v3i.yolo26.zip` a la **raíz de tu Google Drive** (MyDrive).

## 1. Verificar la GPU
Confirmamos que Colab nos asignó una GPU.

In [ ]:
!nvidia-smi

## 2. Instalar Ultralytics (YOLO)

In [ ]:
!pip install ultralytics -q

import ultralytics
ultralytics.checks()   # muestra versión, GPU y entorno

## 3. Montar Google Drive y preparar el dataset

Asegúrate de haber subido `yolo-v8-trash-detection-EE4016.v3i.yolo26.zip` a la
**raíz de tu Google Drive**. Al ejecutar la celda, autoriza el acceso.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
from pathlib import Path

# >>> AJUSTA esta ruta si subiste el ZIP a otra carpeta de tu Drive <<<
ZIP_EN_DRIVE = "/content/drive/MyDrive/yolo-v8-trash-detection-EE4016.v3i.yolo26.zip"
DESTINO = "/content/dataset"

assert os.path.exists(ZIP_EN_DRIVE), f"No encuentro el ZIP en {ZIP_EN_DRIVE}. Súbelo a tu Drive o corrige la ruta."

os.makedirs(DESTINO, exist_ok=True)
with zipfile.ZipFile(ZIP_EN_DRIVE, "r") as z:
    z.extractall(DESTINO)

# Verificación rápida del dataset descomprimido
for split in ["train", "valid", "test"]:
    carpeta = Path(DESTINO) / split / "images"
    n = len(list(carpeta.glob("*.jpg"))) if carpeta.exists() else 0
    print(f"{split:6}: {n} imágenes")

### Ajustar `data.yaml` con rutas absolutas
El `data.yaml` de Roboflow trae rutas relativas (`../train/images`) que suelen
dar el error *"dataset not found"* en Colab. Lo reescribimos con rutas absolutas.

In [ ]:
data_yaml = (
    f"train: {DESTINO}/train/images\n"
    f"val: {DESTINO}/valid/images\n"
    f"test: {DESTINO}/test/images\n"
    "nc: 6\n"
    "names: ['Trash', 'cardboard', 'glass', 'metal', 'paper', 'plastic']\n"
)
YAML_PATH = f"{DESTINO}/data.yaml"
with open(YAML_PATH, "w") as f:
    f.write(data_yaml)

print(data_yaml)

## 4. Entrenar YOLO11n

Usamos **YOLO11 nano** (rápido y ligero) como línea base. En una GPU T4 suele
tardar **~25-45 min** para 50 épocas. Si quieres más precisión y tienes tiempo,
cambia `yolo11n.pt` por `yolo11s.pt`.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")   # <- aquí se elige la versión del modelo

results = model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,                 # early stopping si no mejora en 15 épocas
    name="residuos_yolo11n",
    plots=True,
)

## 5. Evaluar el modelo
Métricas globales sobre el conjunto de validación.

In [ ]:
metrics = model.val()

print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precisión    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")

### Curvas, matriz de confusión y predicciones de ejemplo

In [ ]:
import os
from IPython.display import Image as ColabImage, display

run_dir = "runs/detect/residuos_yolo11n"
for nombre in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    ruta = os.path.join(run_dir, nombre)
    if os.path.exists(ruta):
        print("==>", nombre)
        display(ColabImage(filename=ruta, width=760))

## 6. Guardar y descargar los pesos entrenados

Guardamos `best.pt` (el mejor modelo) en tu Drive como respaldo y lo descargamos
a tu PC. Cópialo luego a la carpeta `models/` del proyecto.

In [ ]:
import shutil
from google.colab import files

best = "runs/detect/residuos_yolo11n/weights/best.pt"

# 1) Copia de seguridad en tu Google Drive
shutil.copy(best, "/content/drive/MyDrive/residuos_best.pt")
print("✅ Guardado en tu Drive como 'residuos_best.pt'")

# 2) Descargar a tu PC
files.download(best)